# Imports, Helpers, Parameters

### Imports

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns
from textwrap import wrap
import numpy as np
import os
import re
import matplotlib.font_manager as fm




### Parameters

In [2]:
main_color = "#3a5f83"

font_dir = r"C:\Users\teddy\Downloads\OAIPR\Technical\AEI Data Other\Lato"

# Loop through every file in the folder
for font_file in os.listdir(font_dir):
    if font_file.lower().endswith(".ttf") and "lato" in font_file.lower():
        font_path = os.path.join(font_dir, font_file)
        fm.fontManager.addfont(font_path)

plt.rcParams["font.family"] = "Lato"
plt.rcParams["font.weight"] = "normal"

palette = sns.color_palette("colorblind")
# Put once at the TOP of your notebook/script (or just tweak this line)
sns.set_context("notebook", font_scale=1.0)  # was 1.2; smaller = less crowded

chart_size = (17, 9)

# Automation By AI Task Coverage

## Load Data

In [3]:
automation_tasks_imputed = pd.read_csv("../data/automation_tasks_imputed.csv")
automation_tasks_imputed

,soc_code_2019,title,task_comp_nat,task_comp_ut,freq_sum_eco,importance,relevance,tot_emp_nat,tot_emp_ut,major_group_code,...,code_or_title_changed,code_and_title_changed,pct_automated_nat,pct_automated_ut,people_automated_nat,people_automated_ut,eco_value_nat,eco_value_ut,ut_share_of_nat,ut_concentration_index
0,11-1011.00,Chief Executives,3.761657e+06,70669.793869,17.756230,3.950968,72.697419,211850.0,3980.0,11,...,False,False,25.784269,51.563014,54623.974815,1026.213924,1.127548e+10,1.682786e+08,0.037570,2.817785
1,11-1011.03,Chief Sustainability Officers,1.618393e+06,30404.547177,7.639333,3.757222,96.502778,211850.0,3980.0,11,...,False,False,0.090152,0.181439,190.986326,3.588037,3.942340e+07,5.883663e+05,0.037810,2.835835
2,11-1021.00,General and Operations Managers,6.987029e+07,894913.317908,19.492775,3.695882,69.391176,3584420.0,45910.0,11,...,False,False,15.180366,15.180366,544128.069174,6969.305956,5.601798e+10,6.358098e+08,0.012808,0.960635
3,11-2011.00,Advertising and Promotions Managers,3.725257e+05,4096.017286,17.655247,3.691905,72.793810,21100.0,232.0,11,...,True,True,27.153128,27.172951,5729.310092,62.995258,7.273932e+08,6.826796e+06,0.011003,0.825263
4,11-2021.00,Marketing Managers,3.934261e+06,67039.208492,10.219392,3.429500,73.021000,384980.0,6560.0,11,...,False,False,62.074767,62.074767,238975.437063,4072.104699,3.848221e+10,5.505078e+08,0.017040,1.278014
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
873,53-7071.00,Gas Compressor and Gas Pumping Station Operators,1.535880e+05,1683.156225,30.056361,3.985385,91.804615,5110.0,56.0,53,...,False,False,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,NaN,NaN
874,53-7072.00,"Pump Operators, Except Wellhead Pumpers",3.101347e+05,4922.773077,24.613865,4.162143,76.587143,12600.0,200.0,53,...,False,False,5.494263,5.494263,692.277080,10.988525,4.155047e+07,8.113927e+05,0.015873,1.190500
875,53-7073.00,Wellhead Pumpers,3.531585e+05,3663.892731,20.354960,4.058571,70.295714,17350.0,180.0,53,...,False,False,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,NaN,NaN
876,53-7081.00,Refuse and Recyclable Material Collectors,5.067620e+06,24759.172431,36.410548,3.975000,82.457857,139180.0,680.0,53,...,False,False,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,NaN,NaN


## Charts

### Workers Automated by Major Occupational Category National

In [ ]:
# Output directory for publication-ready charts
outdir = "../outputs/charts_for_sharing/executive_summary_oct_2025"
os.makedirs(outdir, exist_ok=True)

# Filter: no frequency increase (removes data quality issues)
no_freq_increase = automation_tasks_imputed['freq_sum_ai'] <= automation_tasks_imputed['freq_sum_eco']
subset = automation_tasks_imputed[no_freq_increase].copy()

# Aggregate to major occupation category level
grouped = subset.groupby("major_occ_category").agg({
    "people_automated_nat": "sum",
    "ai_task_comp_nat": "sum",
    "task_comp_nat": "sum"
}).reset_index()

# Compute automation percentages for each category
grouped["pct_automated_nat"] = (grouped["ai_task_comp_nat"] / grouped["task_comp_nat"]) * 100

# Sort by automation percentage and get all categories (there aren't that many)
grouped = grouped.sort_values("people_automated_nat", ascending=False).reset_index(drop=True)

# Create figure
fig, ax = plt.subplots(figsize=chart_size)

# Use same professional color
sns.barplot(data=grouped, x='people_automated_nat', y="major_occ_category", 
            color=main_color, ax=ax)

# Add value labels at end of bars
for i, (idx, row) in enumerate(grouped.iterrows()):
    value = row['people_automated_nat']
    pct = row['pct_automated_nat']
    ax.text(value, i, f' {value:,.0f} ({pct:.1f}%)', 
            va='center', ha='left', fontsize=9)

# Extend x-axis limit slightly to prevent label overlap with border
ax.set_xlim(right=ax.get_xlim()[1] * 1.05)  # Add 5% padding on right side

# Wrap y labels
current_labels = [item.get_text() for item in ax.get_yticklabels()]
wrapped_labels = ['\n'.join(wrap(label, width=50)) for label in current_labels]
ax.set_yticks(ax.get_yticks())
ax.set_yticklabels(wrapped_labels, fontsize=10)
plt.subplots_adjust(left=0.38)

# Professional titles and labels
ax.set_title("Workers Affected by Major Occupational Category (National)", 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel("Number of Workers Affected (% Maj Occ Cat Tasks Affected)", fontsize=12, labelpad=10)
ax.set_ylabel("Major Occupational Category", fontsize=12, labelpad=10)

# Format x-axis as percentage
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))

# Add grid for readability
ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)

plt.tight_layout()

# Save with descriptive filename
filename = "Workers Affected by Major Occupational Category (National).png"
plt.savefig(os.path.join(outdir, filename), dpi=300, bbox_inches='tight')
plt.close(fig)
grouped

,major_occ_category,people_automated_nat,ai_task_comp_nat,task_comp_nat,pct_automated_nat
0,Office and Administrative Support Occupations,6.826476e+06,3.570285e+08,9.833033e+08,36.309093
1,Sales and Related Occupations,3.939629e+06,3.052496e+08,9.856984e+08,30.967847
2,Educational Instruction and Library Occupations,3.702900e+06,2.229687e+08,4.210868e+08,52.950763
3,Business and Financial Operations Occupations,3.629345e+06,6.099254e+07,3.149397e+08,19.366419
4,Management Occupations,3.096247e+06,5.235442e+07,3.099392e+08,16.891836
5,Computer and Mathematical Occupations,1.792107e+06,4.133409e+07,1.378586e+08,29.982966
6,Food Preparation and Serving Related Occupations,1.359983e+06,1.403060e+08,2.057117e+09,6.820517
7,Healthcare Practitioners and Technical Occupat...,1.122916e+06,8.833016e+07,1.798147e+09,4.912288
8,Healthcare Support Occupations,8.361753e+05,4.277991e+07,3.963231e+08,10.794199
9,"Arts, Design, Entertainment, Sports, and Media...",8.290982e+05,2.637534e+07,8.370822e+07,31.508657


### Workers Automated by Major Occupational Category Utah

In [ ]:
# Output directory for publication-ready charts
outdir = "../outputs/charts_for_sharing/executive_summary_oct_2025"
os.makedirs(outdir, exist_ok=True)

# Filter: no frequency increase (removes data quality issues)
no_freq_increase = automation_tasks_imputed['freq_sum_ai'] <= automation_tasks_imputed['freq_sum_eco']
subset = automation_tasks_imputed[no_freq_increase].copy()

# Aggregate to major occupation category level
grouped = subset.groupby("major_occ_category").agg({
    "people_automated_ut": "sum",
    "ai_task_comp_ut": "sum",
    "task_comp_ut": "sum"
}).reset_index()

# Compute automation percentages for each category
grouped["pct_automated_ut"] = (grouped["ai_task_comp_ut"] / grouped["task_comp_ut"]) * 100

# Sort by automation percentage and get all categories (there aren't that many)
grouped = grouped.sort_values("people_automated_ut", ascending=False).reset_index(drop=True)

# Create figure
fig, ax = plt.subplots(figsize=chart_size)

# Use same professional color
sns.barplot(data=grouped, x='people_automated_ut', y="major_occ_category", 
            color=main_color, ax=ax)

# Add value labels at end of bars
for i, (idx, row) in enumerate(grouped.iterrows()):
    value = row['people_automated_ut']
    pct = row['pct_automated_ut']
    ax.text(value, i, f' {value:,.0f} ({pct:.1f}%)', 
            va='center', ha='left', fontsize=9)

# Extend x-axis limit slightly to prevent label overlap with border
ax.set_xlim(right=ax.get_xlim()[1] * 1.02)  # Add 2% padding on right side

# Wrap y labels
current_labels = [item.get_text() for item in ax.get_yticklabels()]
wrapped_labels = ['\n'.join(wrap(label, width=50)) for label in current_labels]
ax.set_yticks(ax.get_yticks())
ax.set_yticklabels(wrapped_labels, fontsize=10)
plt.subplots_adjust(left=0.38)

# Professional titles and labels
ax.set_title("Workers Affected by Major Occupational Category (Utah)", 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel("Number of Workers Affected (% Maj Occ Cat Tasks Affected)", fontsize=12, labelpad=10)
ax.set_ylabel("Major Occupational Category", fontsize=12, labelpad=10)

# Format x-axis as percentage
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))

# Add grid for readability
ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)

plt.tight_layout()

# Save with descriptive filename
filename = "Workers Affected by Major Occupational Category (Utah).png"
plt.savefig(os.path.join(outdir, filename), dpi=300, bbox_inches='tight')
plt.close(fig)

,major_occ_category,people_automated_ut,ai_task_comp_ut,task_comp_ut,pct_automated_ut
0,Office and Administrative Support Occupations,91982.852354,4.771573e+06,1.202615e+07,39.676657
1,Business and Financial Operations Occupations,42061.665842,1.349983e+06,3.443014e+06,39.209345
2,Sales and Related Occupations,40585.531193,3.121266e+06,9.789350e+06,31.884303
3,Educational Instruction and Library Occupations,40029.452315,2.416756e+06,4.417837e+06,54.704506
4,Management Occupations,37881.959396,9.566508e+05,3.444611e+06,27.772393
5,Computer and Mathematical Occupations,23157.788999,1.049227e+06,1.631813e+06,64.298221
6,Food Preparation and Serving Related Occupations,11749.873968,1.676040e+06,2.263729e+07,7.403891
7,Healthcare Practitioners and Technical Occupat...,10370.585481,1.339144e+06,1.521189e+07,8.803267
8,"Arts, Design, Entertainment, Sports, and Media...",10034.172163,3.563073e+05,9.877239e+05,36.073570
9,Healthcare Support Occupations,9421.551584,4.804788e+05,4.203401e+06,11.430717


### Top 15 Workers Automated by Occupation Utah

In [ ]:
# Output directory for publication-ready charts
outdir = "../outputs/charts_for_sharing/executive_summary_oct_2025"
os.makedirs(outdir, exist_ok=True)

# Filter: no frequency increase (removes data quality issues)
no_freq_increase = automation_tasks_imputed['freq_sum_ai'] <= automation_tasks_imputed['freq_sum_eco']
subset = automation_tasks_imputed[no_freq_increase].copy()

# Get top 15 most automated occupations
top_15 = subset.nlargest(15, 'people_automated_ut').copy()

# Create combined label with title and major category
top_15['title_with_category'] = top_15.apply(
    lambda row: f"{row['title']} [{row['major_occ_category']}]",
    axis=1
)

# Create figure
fig, ax = plt.subplots(figsize=chart_size)

# Use a professional color palette
palette = sns.color_palette("colorblind")
sns.barplot(data=top_15, x='people_automated_ut', y='title_with_category', 
            color=main_color, ax=ax)

for i, (idx, row) in enumerate(top_15.iterrows()):
    value = row['people_automated_ut']
    pct = row['pct_automated_nat']
    ax.text(value, i, f' {value:,.0f} ({pct:.1f}%)', 
            va='center', ha='left', fontsize=9)

# Extend x-axis limit slightly to prevent label overlap with border
ax.set_xlim(right=ax.get_xlim()[1] * 1.05)  # Add 5% padding on right side

# Wrap y labels
current_labels = [item.get_text() for item in ax.get_yticklabels()]
wrapped_labels = ['\n'.join(wrap(label, width=65)) for label in current_labels]
ax.set_yticks(ax.get_yticks())
ax.set_yticklabels(wrapped_labels, fontsize=10)
plt.subplots_adjust(left=0.42)

# Professional titles and labels
ax.set_title("Top 15 Most Affected Occupations by Workers (Utah)", 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel("Number of Workers Affected (% Occ Tasks Automated)", fontsize=12, labelpad=10)
ax.set_ylabel("Occupation [Major Occupational Category]", fontsize=12, labelpad=10)

# Format x-axis as percentage
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))

# Add grid for readability
ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)

plt.tight_layout()

# Save with descriptive filename
filename = "Top 15 Most Affected Occupations by Workers (Utah).png"
plt.savefig(os.path.join(outdir, filename), dpi=300, bbox_inches='tight')
plt.close(fig)

top_15

,soc_code_2019,title,task_comp_nat,task_comp_ut,freq_sum_eco,importance,relevance,tot_emp_nat,tot_emp_ut,major_group_code,...,code_and_title_changed,pct_automated_nat,pct_automated_ut,people_automated_nat,people_automated_ut,eco_value_nat,eco_value_ut,ut_share_of_nat,ut_concentration_index,title_with_category
565,43-4051.00,Customer Service Representatives,1.428071e+08,2.554983e+06,52.388414,4.053846,67.595385,2725930.0,48770.0,43,...,False,56.342895,56.342895,1.535868e+06,27478.430101,6.578122e+10,1.103259e+09,0.017891,1.341862,Customer Service Representatives [Office and A...
598,43-9061.00,"Office Clerks, General",1.250238e+08,1.840087e+06,49.799382,3.792000,71.620000,2510550.0,36950.0,43,...,False,46.953886,46.953886,1.178801e+06,17349.460713,5.143108e+10,7.205231e+08,0.014718,1.103863,"Office Clerks, General [Office and Administrat..."
536,41-2031.00,Retail Salespersons,4.091970e+08,4.261829e+06,107.676328,4.190833,71.347083,3800250.0,39580.0,41,...,False,31.780674,31.780674,1.207745e+06,12578.790826,4.176382e+10,4.314525e+08,0.010415,0.781148,Retail Salespersons [Sales and Related Occupat...
551,43-1011.00,First-Line Supervisors of Office and Administr...,3.873861e+07,4.861817e+05,25.902064,3.776429,75.823929,1495580.0,18770.0,43,...,False,49.818247,49.818247,7.450717e+05,9350.885024,4.927905e+10,5.981761e+08,0.012550,0.941292,First-Line Supervisors of Office and Administr...
295,25-2021.00,"Elementary School Teachers, Except Special Edu...",1.418186e+08,1.608209e+06,101.785366,4.135526,94.365789,1393310.0,15800.0,25,...,False,53.203252,53.203252,7.412862e+05,8406.113755,4.621178e+10,5.187413e+08,0.011340,0.850509,"Elementary School Teachers, Except Special Edu..."
2,11-1021.00,General and Operations Managers,6.987029e+07,8.949133e+05,19.492775,3.695882,69.391176,3584420.0,45910.0,11,...,False,15.180366,15.180366,5.441281e+05,6969.305956,5.601798e+10,6.358098e+08,0.012808,0.960635,General and Operations Managers [Management Oc...
298,25-2031.00,"Secondary School Teachers, Except Special and ...",8.661850e+07,9.158202e+05,80.760159,3.945937,93.971563,1072540.0,11340.0,25,...,False,60.536220,60.536220,6.492752e+05,6864.807323,4.193019e+10,5.099179e+08,0.010573,0.792993,"Secondary School Teachers, Except Special and ..."
543,41-4012.00,"Sales Representatives, Wholesale and Manufactu...",6.342059e+07,6.973532e+05,50.061246,4.122222,73.403889,1266860.0,13930.0,41,...,False,48.956444,48.956444,6.202096e+05,6819.632602,4.141760e+10,4.165432e+08,0.010996,0.824693,"Sales Representatives, Wholesale and Manufactu..."
532,41-2011.00,Cashiers,3.353805e+08,2.962783e+06,106.536621,4.351071,65.977143,3148030.0,27810.0,41,...,False,24.299099,24.299099,7.649429e+05,6757.579480,2.385857e+10,2.046871e+08,0.008834,0.662570,Cashiers [Sales and Related Occupations]
99,15-1232.00,Computer User Support Specialists,1.763455e+07,2.719000e+05,25.293022,3.469375,88.588125,697210.0,10750.0,15,...,False,61.949932,61.949932,4.319211e+05,6659.617740,2.606212e+10,3.833276e+08,0.015419,1.156418,Computer User Support Specialists [Computer an...


### Economic Value Generated by Major Occupational Category Utah

In [ ]:
# Output directory for publication-ready charts
outdir = "../outputs/charts_for_sharing/executive_summary_oct_2025"
os.makedirs(outdir, exist_ok=True)

# Filter: no frequency increase (removes data quality issues)
no_freq_increase = automation_tasks_imputed['freq_sum_ai'] <= automation_tasks_imputed['freq_sum_eco']
subset = automation_tasks_imputed[no_freq_increase].copy()

# Aggregate to major occupation category level
grouped = subset.groupby("major_occ_category").agg({
    "eco_value_ut": "sum",
    "ai_task_comp_ut": "sum",
    "task_comp_ut": "sum"
}).reset_index()

# Compute automation percentages for each category
grouped["pct_automated_ut"] = (grouped["ai_task_comp_ut"] / grouped["task_comp_ut"]) * 100

# Sort by economic value and get all categories
grouped = grouped.sort_values("eco_value_ut", ascending=False).reset_index(drop=True)

# Create figure
fig, ax = plt.subplots(figsize=chart_size)

# Use same professional color
sns.barplot(data=grouped, x='eco_value_ut', y="major_occ_category", 
            color=main_color, ax=ax)

# Add labels with dollar amount (in billions) and percentage
for i, (idx, row) in enumerate(grouped.iterrows()):
    value = row['eco_value_ut'] / 1e9  # convert to billions
    pct = row['pct_automated_ut']
    ax.text(row['eco_value_ut'], i, f' ${value:,.1f}B ({pct:.1f}%)', 
            va='center', ha='left', fontsize=9)
    
# Extend x-axis limit slightly to prevent label overlap with border
ax.set_xlim(right=ax.get_xlim()[1] * 1.1)

# Wrap y labels
current_labels = [item.get_text() for item in ax.get_yticklabels()]
wrapped_labels = ['\n'.join(wrap(label, width=50)) for label in current_labels]
ax.set_yticks(ax.get_yticks())
ax.set_yticklabels(wrapped_labels, fontsize=10)
plt.subplots_adjust(left=0.38)

# Professional titles and labels
ax.set_title("Wages Generated by Major Occupational Category (Utah)", 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel("Wages Generated (% Maj Occ Cat Automated)", fontsize=12, labelpad=10)
ax.set_ylabel("Major Occupational Category", fontsize=12, labelpad=10)

# Format x-axis in billions
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x/1e9:,.1f}B"))

# Add grid for readability
ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)

plt.tight_layout()

# Save with descriptive filename
filename = "Wages Generated by Major Occupational Category (Utah).png"
plt.savefig(os.path.join(outdir, filename), dpi=300, bbox_inches='tight')
plt.close(fig)

grouped

,major_occ_category,eco_value_ut,ai_task_comp_ut,task_comp_ut,pct_automated_ut
0,Management Occupations,4.487995e+09,9.566508e+05,3.444611e+06,27.772393
1,Office and Administrative Support Occupations,4.064051e+09,4.771573e+06,1.202615e+07,39.676657
2,Business and Financial Operations Occupations,3.063054e+09,1.349983e+06,3.443014e+06,39.209345
3,Educational Instruction and Library Occupations,2.669708e+09,2.416756e+06,4.417837e+06,54.704506
4,Computer and Mathematical Occupations,2.047232e+09,1.049227e+06,1.631813e+06,64.298221
5,Sales and Related Occupations,1.839385e+09,3.121266e+06,9.789350e+06,31.884303
6,Healthcare Practitioners and Technical Occupat...,1.098284e+09,1.339144e+06,1.521189e+07,8.803267
7,Architecture and Engineering Occupations,6.356983e+08,2.990973e+05,1.409076e+06,21.226484
8,"Arts, Design, Entertainment, Sports, and Media...",6.011335e+08,3.563073e+05,9.877239e+05,36.073570
9,"Installation, Maintenance, and Repair Occupations",5.242036e+08,3.132122e+05,2.573718e+06,12.169643


### Total Utah Economic Value

In [26]:
# Load the data
eco_2025 = pd.read_csv('../data/ratings_eco_2025.csv')

# Get one row per occupation (since employment/wage are same across all tasks for an occupation)
occupation_level = eco_2025.drop_duplicates(subset='title').copy()

# Compute total payroll (wage * employment)
occupation_level['total_payroll_ut'] = occupation_level['tot_emp_ut'] * occupation_level['a_med_ut']

# Sum total employment and total payroll across all occupations
total_payroll_ut = occupation_level['total_payroll_ut'].sum()

# Optional: sort by total payroll to see which occupations contribute most
occupation_level_sorted = occupation_level.sort_values('total_payroll_ut', ascending=False)

# Display results
print(f"Total Utah Payroll (Employment × Wage): ${total_payroll_ut:,.0f}")

# Optional preview
occupation_level_sorted[['title', 'tot_emp_ut', 'a_med_ut', 'total_payroll_ut']].head()


Total Utah Payroll (Employment × Wage): $133,941,368,538


,title,tot_emp_ut,a_med_ut,total_payroll_ut
49,General and Operations Managers,45910.0,91230.0,4.188369e+09
8043,Registered Nurses,25780.0,82270.0,2.120921e+09
7992,Advanced Practice Psychiatric Nurses,25780.0,82270.0,2.120921e+09
8008,Clinical Nurse Specialists,25780.0,82270.0,2.120921e+09
8045,Acute Care Nurses,25780.0,82270.0,2.120921e+09
